# Simulation 2 — Directional coupler transmission vs gap

Two parallel waveguides exchange power via **evanescent coupling**. In coupled-mode theory the cross-port power scales as $|\\sin(\\kappa L)|^2$ where $\\kappa$ grows as the edge gap shrinks. This notebook runs **3D FDTD** and compares simulated splitting to the phenomenological CMT curve from [PIC-component-Library](https://github.com/Shahbaz-z/PIC-component-Library).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import tidy3d.web as web

from fdtd_pic.analytics.cmt import cross_power_curve
from fdtd_pic.config import COUPLING_LENGTH, GAP_SWEEP
from fdtd_pic.coupler import build_coupler_simulation, coupling_ratio, extract_port_powers
from fdtd_pic.plotting import apply_style, save_figure
from fdtd_pic.sweep import sweep_param

apply_style()

In [ ]:
# Single-gap sanity check — always plot geometry before cloud submit
gap_test = 0.2
sim = build_coupler_simulation(gap=gap_test)
sim.plot(z=0)
plt.title(f'Coupler geometry, gap={gap_test} um')
plt.show()

In [ ]:
# Run one job first (or load from cache)
sim_data = web.run(sim, task_name='coupler_gap0.2', verbose=True)
powers = extract_port_powers(sim_data)
print(powers)
print('Coupling ratio:', coupling_ratio(sim_data))
print('Output sum:', powers.total_output)

In [ ]:
results = sweep_param(
    build_coupler_simulation,
    param_name='gap',
    values=GAP_SWEEP,
    task_prefix='coupler',
)
ratios = [coupling_ratio(results[g]) for g in GAP_SWEEP]
cmt = cross_power_curve(GAP_SWEEP, COUPLING_LENGTH)

fig, ax = plt.subplots()
ax.plot(GAP_SWEEP, ratios, 'o-', label='Tidy3D FDTD')
ax.plot(GAP_SWEEP, cmt, '--', label='CMT (phenomenological)')
ax.set_xlabel('Gap (um)')
ax.set_ylabel('Cross power fraction')
ax.set_title(f'Directional coupler, L={COUPLING_LENGTH} um')
ax.legend()
save_figure(fig, '../assets/coupler_ratio_vs_gap.png')
plt.show()

## Key takeaway

Smaller gap increases coupling, but at fixed $L$ the cross power is **not monotonic** — it oscillates as $\\kappa L$ advances. FDTD validates layout knobs from Project 1.